## Imports

In [1]:
import os
import json
import random
import warnings
from pathlib import Path
import itertools

import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier, log_evaluation, early_stopping
from sklearn.model_selection import KFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import balanced_accuracy_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 200)

## CFG

In [2]:
class CFG:
    EXP_ID = "exp_20260417_047b_lgb_digit_te_min_with_optuna"
    EXP_NAME = "lgb_digit_te_min_with_optuna"
    SAVE_DIR = Path(f"/kaggle/working/{EXP_ID}")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"

    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"
    SEED = 2026
    N_FOLDS = 5
    NUM_CLASSES = 3

    LGB_PARAMS = {
        "n_estimators": 6000,
        "boosting_type": "gbdt",
        "max_depth": 4,
        "num_leaves": 32,
        "learning_rate": 0.05,
        "feature_fraction": 0.6,
        "bagging_fraction": 0.7,
        "bagging_freq": 1,
        "lambda_l1": 10,
        "lambda_l2": 10,
        "min_child_samples": 12,
        "random_state": 2026,
        "n_jobs": -1,
        "max_bin": 15000,
        "verbosity": -1,
        "subsample": 0.5,
        "subsample_for_bin": 100000,
        "subsample_freq": 1,
    }

    best_params_payload = {}
    best_params_path = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/exp_20260416_047a_lgb025_optuna_resume/best_params.json"
    with open(best_params_path, mode="rt", encoding="utf-8") as f:
        best_params_payload = json.load(f)

    LGB_PARAMS.update(best_params_payload["best_params"])

## utility

In [3]:
def seed_everything(seed):
    np.random.seed(seed)
    random.seed(seed)

def balanced_acc_score_mc(y_true, y_pred):
    if len(y_pred.shape) == 2:
        y_pred = np.argmax(y_pred, axis=1)
    c = len(np.unique(y_true))
    acc = 0.0
    for i in range(c):
        acc += np.sum((y_true == i) & (y_pred == i)) / np.sum(y_true == i) / c
    return acc

def lgb_eval_metric(y_true, y_pred):
    score = balanced_acc_score_mc(y_true, y_pred)
    return "acc", score, True

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def safe_logit_like(proba, eps=1e-12):
    return np.log(np.clip(proba, eps, 1.0))

def apply_bias(proba, bias):
    """
    bias: iterable of len 3
    apply in log-prob space, then renormalize
    """
    logp = safe_logit_like(proba)
    adjusted = logp + np.array(bias, dtype=np.float32).reshape(1, -1)

    adjusted = adjusted - adjusted.max(axis=1, keepdims=True)
    expv = np.exp(adjusted)
    out = expv / expv.sum(axis=1, keepdims=True)
    return out.astype(np.float32)

def score_bias(proba, y_true, bias):
    adj = apply_bias(proba, bias)
    pred = np.argmax(adj, axis=1)
    return balanced_accuracy_score(y_true, pred)

def run_grid_search(proba, y_true, bias_ranges):
    best_bias = None
    best_score = -1.0

    for b0, b1, b2 in itertools.product(*bias_ranges):
        bias = (b0, b1, b2)
        sc = score_bias(proba, y_true, bias)
        if sc > best_score:
            best_score = sc
            best_bias = bias

    return best_bias, float(best_score)


seed_everything(CFG.SEED)

## load data

In [4]:
drop_cols = [CFG.ID_COL]

train = pd.read_csv(CFG.TRAIN_PATH).drop(drop_cols, axis=1)
test = pd.read_csv(CFG.TEST_PATH).drop(drop_cols, axis=1)

print(f"train.shape={train.shape}, test.shape={test.shape}")

target2idx = {v: i for i, v in enumerate(train[CFG.TARGET_COL].unique())}
idx2target = {v: k for k, v in target2idx.items()}

train[CFG.TARGET_COL] = train[CFG.TARGET_COL].map(target2idx)

CATS = [c for c in test.columns if train[c].dtype == object]
NUMS = [c for c in test.columns if c not in CATS]

print("CATS:", CATS)
print("NUMS:", NUMS)
display(train.head())

train.shape=(630000, 20), test.shape=(270000, 19)
CATS: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
NUMS: ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']


,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,0
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,0
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,0
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,1
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,0


## digit FE

In [5]:
M = train[NUMS].max()

def add_digit_features(df):
    df = df.copy()

    for c in NUMS:
        for k in range(-4, 4):
            df[f"{c}_digit{k}"] = (df[c] // (10 ** k) % 10).astype("int8")

        if M[c] < 10:
            df[c] = df[c].round(3)
        elif M[c] < 100:
            df[c] = df[c].round(2)
        else:
            df[c] = df[c].round(1)

    return df

train = add_digit_features(train)
test = add_digit_features(test)

DROP = [c for c in test.columns if test[c].nunique() == 1]
print("DROP:", DROP)

train.drop(DROP, axis=1, inplace=True)
test.drop(DROP, axis=1, inplace=True)

DROP: ['Soil_pH_digit1', 'Soil_pH_digit2', 'Soil_pH_digit3', 'Soil_Moisture_digit2', 'Soil_Moisture_digit3', 'Organic_Carbon_digit1', 'Organic_Carbon_digit2', 'Organic_Carbon_digit3', 'Electrical_Conductivity_digit1', 'Electrical_Conductivity_digit2', 'Electrical_Conductivity_digit3', 'Temperature_C_digit2', 'Temperature_C_digit3', 'Humidity_digit2', 'Humidity_digit3', 'Sunlight_Hours_digit2', 'Sunlight_Hours_digit3', 'Wind_Speed_kmh_digit2', 'Wind_Speed_kmh_digit3', 'Field_Area_hectare_digit2', 'Field_Area_hectare_digit3', 'Previous_Irrigation_mm_digit3']


## category remap

In [6]:
CATEGORY = CATS + [c for c in test.columns if "digit" in c]

for c in CATEGORY:
    freq = train[c].value_counts()
    mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
    mapping_default = len(mapping)

    train[c] = train[c].map(lambda x: mapping.get(x, mapping_default))
    test[c] = test[c].map(lambda x: mapping.get(x, mapping_default))

FEATURES = CATEGORY + NUMS

print("n_CATEGORY =", len(CATEGORY))
print("n_FEATURES =", len(FEATURES))
display(train[FEATURES + [CFG.TARGET_COL]].head())

n_CATEGORY = 74
n_FEATURES = 85


,Soil_Type,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Mulching_Used,Region,Soil_pH_digit-4,Soil_pH_digit-3,Soil_pH_digit-2,Soil_pH_digit-1,Soil_pH_digit0,Soil_Moisture_digit-4,Soil_Moisture_digit-3,Soil_Moisture_digit-2,Soil_Moisture_digit-1,Soil_Moisture_digit0,Soil_Moisture_digit1,Organic_Carbon_digit-4,Organic_Carbon_digit-3,Organic_Carbon_digit-2,Organic_Carbon_digit-1,Organic_Carbon_digit0,Electrical_Conductivity_digit-4,Electrical_Conductivity_digit-3,Electrical_Conductivity_digit-2,Electrical_Conductivity_digit-1,Electrical_Conductivity_digit0,Temperature_C_digit-4,Temperature_C_digit-3,Temperature_C_digit-2,Temperature_C_digit-1,Temperature_C_digit0,Temperature_C_digit1,Humidity_digit-4,Humidity_digit-3,Humidity_digit-2,Humidity_digit-1,Humidity_digit0,Humidity_digit1,Rainfall_mm_digit-4,Rainfall_mm_digit-3,Rainfall_mm_digit-2,Rainfall_mm_digit-1,Rainfall_mm_digit0,Rainfall_mm_digit1,Rainfall_mm_digit2,Rainfall_mm_digit3,Sunlight_Hours_digit-4,Sunlight_Hours_digit-3,Sunlight_Hours_digit-2,Sunlight_Hours_digit-1,Sunlight_Hours_digit0,Sunlight_Hours_digit1,Wind_Speed_kmh_digit-4,Wind_Speed_kmh_digit-3,Wind_Speed_kmh_digit-2,Wind_Speed_kmh_digit-1,Wind_Speed_kmh_digit0,Wind_Speed_kmh_digit1,Field_Area_hectare_digit-4,Field_Area_hectare_digit-3,Field_Area_hectare_digit-2,Field_Area_hectare_digit-1,Field_Area_hectare_digit0,Field_Area_hectare_digit1,Previous_Irrigation_mm_digit-4,Previous_Irrigation_mm_digit-3,Previous_Irrigation_mm_digit-2,Previous_Irrigation_mm_digit-1,Previous_Irrigation_mm_digit0,Previous_Irrigation_mm_digit1,Previous_Irrigation_mm_digit2,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Field_Area_hectare,Previous_Irrigation_mm,Irrigation_Need
0,2,0,3,2,3,3,0,2,0,0,4,1,4,0,0,5,6,2,3,0,0,5,9,1,0,0,2,9,3,0,0,5,4,3,2,0,0,0,2,9,1,0,0,2,2,4,6,6,1,1,1,8,0,0,0,0,0,3,3,7,0,0,0,8,3,4,0,0,0,0,6,3,0,1,4.92,32.58,1.01,3.05,15.01,50.61,726.0,5.90,16.79,0.82,112.2,0
1,1,4,2,0,2,1,1,0,0,0,8,2,2,0,0,1,3,7,1,0,0,6,0,0,0,0,9,4,0,1,1,2,1,0,1,0,0,4,9,3,3,0,0,0,0,4,8,9,1,1,1,2,0,6,0,0,1,9,1,1,1,0,0,1,6,6,0,0,0,0,6,5,5,0,7.08,56.61,0.44,2.00,22.92,67.86,985.7,6.98,3.39,5.27,47.2,0
2,1,1,2,0,1,0,1,4,1,1,9,0,1,0,1,9,7,8,2,1,1,8,6,0,0,1,4,8,0,0,0,7,1,9,1,0,0,1,1,5,6,0,0,3,0,1,9,3,2,0,0,1,3,6,0,0,1,4,0,1,1,0,1,6,6,8,0,0,0,6,8,2,0,1,5.69,27.71,0.81,2.83,26.97,92.22,2201.7,6.05,3.85,8.24,110.4,0
3,0,4,1,0,0,1,1,0,1,1,5,0,1,0,1,2,0,5,4,1,1,6,2,1,0,0,5,8,2,0,1,2,6,8,2,0,0,9,6,0,3,0,0,9,4,3,3,2,0,0,0,4,5,2,0,0,1,0,1,5,1,0,1,5,4,8,0,0,1,0,9,4,3,0,5.65,13.32,1.33,0.87,13.32,61.57,1357.3,9.12,2.31,8.32,53.8,1
4,1,4,3,1,0,1,0,0,0,0,5,1,2,0,0,4,2,4,1,0,0,0,2,0,0,0,3,4,2,0,0,1,9,1,1,0,0,0,8,0,6,0,1,1,8,5,2,8,0,0,1,3,0,6,0,0,0,2,5,1,0,0,0,1,4,7,0,0,0,9,6,4,4,0,7.96,59.14,0.38,0.96,20.22,91.11,1538.2,6.95,13.94,7.37,93.2,0


## class weight

In [7]:
unique, counts = np.unique(train[CFG.TARGET_COL].values, return_counts=True)
count_dict = dict(zip(unique, counts))
avg_count = len(train) / len(unique)
weights_dict = {cls: avg_count / cnt for cls, cnt in count_dict.items()}
sample_weights = np.array([weights_dict[y] for y in train[CFG.TARGET_COL]])

print("weights_dict =", weights_dict)

weights_dict = {np.int64(0): np.float64(0.5676949153458749), np.int64(1): np.float64(0.8783891180136694), np.int64(2): np.float64(9.995716121662145)}


## CV main

In [8]:
X = train.drop([CFG.TARGET_COL], axis=1)
y = train[CFG.TARGET_COL]
test_X = test.copy()

oof_preds = np.zeros((len(y), CFG.NUM_CLASSES))
test_preds = np.zeros((len(test_X), CFG.NUM_CLASSES))

kf = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=42)
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    print(f"\nFold {fold}/{CFG.N_FOLDS}")

    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()
    train_weights = sample_weights[train_idx]

    te = TargetEncoder(target_type="multiclass", smooth="auto", cv=5, random_state=42)

    X_train_enc = te.fit_transform(X_train[FEATURES], y_train)
    X_val_enc = te.transform(X_val[FEATURES])
    X_test_enc = te.transform(test_X[FEATURES])

    X_train_enc = pd.DataFrame(X_train_enc, index=X_train.index)
    X_val_enc = pd.DataFrame(X_val_enc, index=X_val.index)
    X_test_enc = pd.DataFrame(X_test_enc, index=test_X.index)

    X_train = pd.concat([X_train, X_train_enc], axis=1)
    X_val = pd.concat([X_val, X_val_enc], axis=1)
    X_test = pd.concat([test_X, X_test_enc], axis=1)

    # shared code 準拠: 元カテゴリ列は drop
    X_train = X_train.drop(CATS, axis=1)
    X_val = X_val.drop(CATS, axis=1)
    X_test = X_test.drop(CATS, axis=1)

    model = LGBMClassifier(**CFG.LGB_PARAMS)
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        sample_weight=train_weights,
        eval_metric=lgb_eval_metric,
        callbacks=[log_evaluation(100), early_stopping(250)],
    )

    y_pred = model.predict_proba(X_val)

    oof_preds[val_idx] = y_pred
    test_preds += model.predict_proba(X_test) / CFG.N_FOLDS

    fold_ba = balanced_acc_score_mc(y_val.values, y_pred)
    fold_scores.append(float(fold_ba))
    print(f"fold_ba = {fold_ba:.6f}")


Fold 1/5
Training until validation scores don't improve for 250 rounds
[100]	valid_0's multi_logloss: 0.0737638	valid_0's acc: 0.976276
[200]	valid_0's multi_logloss: 0.0575928	valid_0's acc: 0.978283
[300]	valid_0's multi_logloss: 0.0529606	valid_0's acc: 0.978479
[400]	valid_0's multi_logloss: 0.0506026	valid_0's acc: 0.977728
[500]	valid_0's multi_logloss: 0.049228	valid_0's acc: 0.977609
Early stopping, best iteration is:
[299]	valid_0's multi_logloss: 0.0529815	valid_0's acc: 0.978541
fold_ba = 0.978541

Fold 2/5
Training until validation scores don't improve for 250 rounds
[100]	valid_0's multi_logloss: 0.0727058	valid_0's acc: 0.977989
[200]	valid_0's multi_logloss: 0.0568759	valid_0's acc: 0.979941
[300]	valid_0's multi_logloss: 0.0524546	valid_0's acc: 0.97983
[400]	valid_0's multi_logloss: 0.0502333	valid_0's acc: 0.979552
[500]	valid_0's multi_logloss: 0.0488909	valid_0's acc: 0.979654
Early stopping, best iteration is:
[259]	valid_0's multi_logloss: 0.0538454	valid_0's acc

## raw CV

In [9]:
raw_cv_ba = balanced_acc_score_mc(y.values, oof_preds)
print(f"raw_cv_ba = {raw_cv_ba:.6f}")

raw_cv_ba = 0.979341


## bias search

In [10]:
oof_raw = oof_preds.copy()
pred_raw = test_preds.copy()

In [11]:
# ---------------------------------------------------------
# Stage 1: coarse
# ---------------------------------------------------------
coarse_vals = np.arange(-1.0, 1.01, 0.1)
best_bias_1, best_score_1 = run_grid_search(
    oof_raw, y,
    [coarse_vals, coarse_vals, coarse_vals]
)
print("stage1 best_bias:", best_bias_1, "score:", best_score_1)

stage1 best_bias: (np.float64(-0.9), np.float64(-1.0), np.float64(-1.0)) score: 0.9793583390173576


In [12]:
# ---------------------------------------------------------
# Stage 2: local refine
# ---------------------------------------------------------
b0, b1, b2 = best_bias_1
local_vals_0 = np.arange(b0 - 0.10, b0 + 0.1001, 0.02)
local_vals_1 = np.arange(b1 - 0.10, b1 + 0.1001, 0.02)
local_vals_2 = np.arange(b2 - 0.10, b2 + 0.1001, 0.02)

best_bias_2, best_score_2 = run_grid_search(
    oof_raw, y,
    [local_vals_0, local_vals_1, local_vals_2]
)
print("stage2 best_bias:", best_bias_2, "score:", best_score_2)

stage2 best_bias: (np.float64(-1.0), np.float64(-1.1), np.float64(-1.1)) score: 0.9793583390173576


In [13]:
# ---------------------------------------------------------
# Stage 3: ultra-local refine
# ---------------------------------------------------------
b0, b1, b2 = best_bias_2
ultra_vals_0 = np.arange(b0 - 0.02, b0 + 0.0201, 0.005)
ultra_vals_1 = np.arange(b1 - 0.02, b1 + 0.0201, 0.005)
ultra_vals_2 = np.arange(b2 - 0.02, b2 + 0.0201, 0.005)

best_bias_3, best_score_3 = run_grid_search(
    oof_raw, y,
    [ultra_vals_0, ultra_vals_1, ultra_vals_2]
)
print("stage3 best_bias:", best_bias_3, "score:", best_score_3)

final_bias = best_bias_3
final_cv = best_score_3

oof_biased = apply_bias(oof_raw, final_bias)
pred_biased = apply_bias(pred_raw, final_bias)

np.save(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_biased_refined.npy", oof_biased)
np.save(CFG.SAVE_DIR / f"pred_{CFG.EXP_NAME}_biased_refined.npy", pred_biased)
np.save(CFG.SAVE_DIR / "best_class_bias_refined.npy", np.array(final_bias, dtype=np.float32))

final_test_preds = np.argmax(pred_biased, axis=1)

submission = pd.read_csv(
    "/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv")
submission[CFG.TARGET_COL] = pd.Series(final_test_preds).map(idx2target)
submission.to_csv(CFG.SAVE_DIR / f"submission_{CFG.EXP_NAME}.csv", index=False)

cv_result = {
    "exp_id": CFG.EXP_ID,
    "base_parent": "exp_20260409_025_lgb_digit_te_min",
    "raw_cv": float(raw_cv_ba),
    "refined_biased_cv": float(final_cv),
    "best_bias": list(map(float, final_bias)),
    "search": {
        "stage1": {"range": [-1.0, 1.0], "step": 0.1},
        "stage2": {"center": list(map(float, best_bias_1)), "width": 0.10, "step": 0.02},
        "stage3": {"center": list(map(float, best_bias_2)), "width": 0.02, "step": 0.005},
    }
}
save_json(cv_result, CFG.SAVE_DIR / "cv_result.json")

print("final_bias:", final_bias)
print("final_cv:", final_cv)
print("saved to:", CFG.SAVE_DIR)

stage3 best_bias: (np.float64(-1.0150000000000001), np.float64(-1.12), np.float64(-1.1050000000000004)) score: 0.9793621369507131
final_bias: (np.float64(-1.0150000000000001), np.float64(-1.12), np.float64(-1.1050000000000004))
final_cv: 0.9793621369507131
saved to: /kaggle/working/exp_20260417_047b_lgb_digit_te_min_with_optuna
